In [ ]:
# python-dotenv
!pip install phidata groq chromadb pypdf sentence-transformers

In [ ]:
# !pip install -U phidata chromadb sentence-transformers pypdf

In [ ]:
from phi.embedder.huggingface import HuggingfaceCustomEmbedder
from phi.embedder.sentence_transformer import SentenceTransformerEmbedder
from phi.knowledge.pdf import PDFUrlKnowledgeBase
from phi.vectordb.chroma import ChromaDb
from phi.llm.groq import Groq
from phi.assistant import Assistant

In [ ]:
# hf_embedder = HuggingfaceCustomEmbedder(model="sentence-transformers/all-mpnet-base-v2")
local_embedder = SentenceTransformerEmbedder()

### **RAG Pipeline**
1. Download PDF
2. Extract text
3. Split into chunks
4. Send each chunk to embedder
5. Convert chunk into Vector
5. Stored in Chroma

In [ ]:
knowledge_base = PDFUrlKnowledgeBase(
    urls=["https://phi-public.s3.amazonaws.com/recipes/ThaiRecipes.pdf"],
    vector_db=ChromaDb(
        collection="thai_recipes",
        embedder=local_embedder
    ),
    # Reader-specific settings to handle slow PDF downloads
    reader_params={"timeout": 60}
)

In [ ]:
try:
    knowledge_base.load(recreate=False)
    print("✅ Knowledge base loaded successfully!")
except Exception as e:
    print(f"❌ Loading failed: {e}")
    print("Try running the cell again; often a second attempt succeeds after partial downloads.")

### **Assistant Setup**

User query → search vector DB → send context to LLM

In [ ]:
assistant = Assistant(
    llm=Groq(model="llama-3.1-8b-instant"),
    knowledge_base=knowledge_base,
    search_knowledge=True,
    show_tool_calls=True,
    stream=False,
    instructions=["Answer only from the PDF. Be concise."]
)

### **Chat Function**
1. Convert query → embedding
2. Search ChromaDB
3. Retrieve relevant chunks
4. Send to Groq LLM
5. Generate answer

In [ ]:
def chat():
    print("🤖 PDF Assistant Ready! Type 'exit' to stop.\n")

    while True:
        query = input("You: ")

        if query.lower() == "exit":
            break

        response = assistant.run(query)

        final_response = ""
        for chunk in response:
            final_response += chunk

        print("\nAssistant:", final_response, "\n")

chat()